# Manufacturing: Predictive Maintenance — End-to-End Machine Learning
**Audience:** beginner to senior ML engineer. **Dataset:** reproducible synthetic data, not real-world evidence. **Goal:** binary classification with statistics, model comparison, tuning and saved inference.

Run from the repository root with `jupyter notebook` and choose **Run All**. Do not use healthcare outputs for patient care or production decisions.


## 1. Environment and imports
We use a reproducible random seed, pandas for tabular analysis, SciPy for statistical inference and scikit-learn pipelines to avoid preprocessing leakage.


In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay, precision_recall_curve
from sklearn.inspection import permutation_importance
from joblib import dump, load
from src.project_utils import make_domain_data, score_classifier
SEED = 42
pd.set_option('display.max_columns', 30)
print('Ready:', ROOT)


## 2. Generate synthetic domain data
The target is generated probabilistically from multiple noisy signals. These are artificial associations, not validated scientific effects. Re-run with the same seed for identical rows.


In [ ]:
df, target, description = make_domain_data('manufacturing', n=2400, seed=SEED)
print(description)
display(df.head())
print('Rows/columns:',df.shape, 'Positive rate:',round(df[target].mean(),3))
(ROOT/'data').mkdir(exist_ok=True)
df.to_csv(ROOT/'data'/'manufacturing_synthetic.csv',index=False)


## 3. Data quality and exploratory analysis
Check duplicates, nulls, types, class imbalance and numeric distributions before modeling.


In [ ]:
display(df.describe(include='all').T)
print('Missing values:',df.isna().sum().to_dict())
print('Duplicates:',df.duplicated().sum())
fig,axes=plt.subplots(1,2,figsize=(11,4))
df[target].value_counts().sort_index().plot.bar(ax=axes[0],title='Class distribution')
df.select_dtypes('number').drop(columns=[target]).hist(ax=None,figsize=(11,7),bins=22)
plt.tight_layout();plt.show()
plt.close(fig)


## 4. Descriptive statistics and confidence intervals
Mean, median, standard deviation, IQR, skewness and 95% t confidence interval describe the numeric sample. A CI concerns the mean under sampling assumptions; it is not an individual prediction interval.


In [ ]:
numeric = df.select_dtypes(include='number').drop(columns=[target])
summary = pd.DataFrame({'mean':numeric.mean(),'median':numeric.median(),'std':numeric.std(),'iqr':numeric.quantile(.75)-numeric.quantile(.25),'skew':numeric.skew()})
summary['ci_low']=numeric.apply(lambda s: stats.t.interval(.95,len(s)-1,loc=s.mean(),scale=stats.sem(s))[0])
summary['ci_high']=numeric.apply(lambda s: stats.t.interval(.95,len(s)-1,loc=s.mean(),scale=stats.sem(s))[1])
display(summary.round(3))


## 5. Hypothesis tests and effect size
Welch t-test compares feature means between outcome groups without assuming equal variances. Mann–Whitney tests distributional rank differences. Chi-square tests categorical association. P-values are exploratory, not proof of causality; many tests increase false positives.


In [ ]:
feature=numeric.columns[0]
a=df.loc[df[target]==0,feature]; b=df.loc[df[target]==1,feature]
print('Welch t-test:',stats.ttest_ind(a,b,equal_var=False))
print('Mann-Whitney U:',stats.mannwhitneyu(a,b,alternative='two-sided'))
pooled=np.sqrt(((len(a)-1)*a.var()+(len(b)-1)*b.var())/(len(a)+len(b)-2))
print('Cohen d:',(b.mean()-a.mean())/pooled)
category=df.select_dtypes('object').columns[0]
contingency=pd.crosstab(df[category],df[target]);display(contingency)
print('Chi-square:',stats.chi2_contingency(contingency)[:2])


## 6. Correlation, normality and outliers
Pearson measures linear association; Spearman measures rank association. Shapiro–Wilk is sensitive with large samples, so inspect plots too. IQR outliers are flagged, not automatically deleted.


In [ ]:
display(numeric.corr(method='pearson').round(2))
display(numeric.corr(method='spearman').round(2))
print('Shapiro p (sample of 300):',stats.shapiro(numeric[feature].sample(300,random_state=SEED)).pvalue)
q1,q3=numeric[feature].quantile([.25,.75]);iqr=q3-q1
print('IQR outlier count:',((numeric[feature]<q1-1.5*iqr)|(numeric[feature]>q3+1.5*iqr)).sum())
plt.figure(figsize=(7,4));plt.scatter(numeric[feature],df[target]+np.random.default_rng(SEED).normal(0,.04,len(df)),alpha=.18,s=9)
plt.title('Feature vs target (jittered)');plt.xlabel(feature);plt.ylabel(target);plt.show()


## 7. Split BEFORE preprocessing
Stratified train/test split preserves rare class proportions. Imputation, scaling and one-hot encoding are fitted only on training folds via Pipeline. Test data remains untouched until final evaluation.


In [ ]:
X=df.drop(columns=[target]); y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=SEED,stratify=y)
num_cols=X.select_dtypes(include='number').columns.tolist()
cat_cols=X.select_dtypes(exclude='number').columns.tolist()
preprocess=ColumnTransformer([
 ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols),
 ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat_cols)
],sparse_threshold=0)
print('Train:',X_train.shape,'Test:',X_test.shape,'Fraud/fault rate:',y.mean())


## 8. Model zoo and stratified cross-validation
Compare a dummy baseline, interpretable linear model, probabilistic model, distance-based model, tree ensembles, boosting and SVM. CV scoring uses average precision (PR-AUC), useful for imbalanced binary outcomes. This is a representative model selection, not literally every algorithm.


In [ ]:
models={
 'Dummy':DummyClassifier(strategy='prior'),
 'LogisticRegression':LogisticRegression(max_iter=1000,class_weight='balanced',random_state=SEED),
 'GaussianNB':GaussianNB(),
 'KNN':KNeighborsClassifier(n_neighbors=15),
 'DecisionTree':DecisionTreeClassifier(max_depth=6,class_weight='balanced',random_state=SEED),
 'RandomForest':RandomForestClassifier(n_estimators=100,min_samples_leaf=3,class_weight='balanced',random_state=SEED,n_jobs=-1),
 'ExtraTrees':ExtraTreesClassifier(n_estimators=100,min_samples_leaf=3,class_weight='balanced',random_state=SEED,n_jobs=-1),
 'GradientBoosting':GradientBoostingClassifier(random_state=SEED),
 'HistGradientBoosting':HistGradientBoostingClassifier(max_iter=80,random_state=SEED),
 'SVM':SVC(kernel='rbf',probability=True,class_weight='balanced',random_state=SEED)
}
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
cv_rows=[]
for name,model in models.items():
    pipe=Pipeline([('prep',preprocess),('model',model)])
    scores=cross_validate(pipe,X_train,y_train,cv=cv,scoring={'pr_auc':'average_precision','roc_auc':'roc_auc'},n_jobs=1)
    cv_rows.append({'model':name,'cv_pr_auc':scores['test_pr_auc'].mean(),'cv_roc_auc':scores['test_roc_auc'].mean()})
cv_results=pd.DataFrame(cv_rows).sort_values('cv_pr_auc',ascending=False)
display(cv_results.round(3))


## 9. Hyperparameter search on training data
Tune the Random Forest using randomized cross-validation; optimize average precision. This is deliberately small for local laptops. Other models remain comparable in the CV table.


In [ ]:
rf_pipe=Pipeline([('prep',preprocess),('model',RandomForestClassifier(random_state=SEED,n_jobs=-1,class_weight='balanced'))])
search=RandomizedSearchCV(rf_pipe,{'model__n_estimators':[60,100,140],'model__max_depth':[4,7,None],'model__min_samples_leaf':[2,5,10]},n_iter=6,scoring='average_precision',cv=cv,random_state=SEED,n_jobs=1)
search.fit(X_train,y_train)
print('RF best params:',search.best_params_,'CV PR-AUC:',round(search.best_score_,3))


## 10. Holdout evaluation and threshold selection
Choose the winner by training CV, not by looking at test scores. Evaluate once on held-out data. For imbalanced targets, accuracy can be misleading. Threshold is illustrated at 0.5; tune it on a separate validation fold in real deployments.


In [ ]:
winner_name=cv_results.iloc[0]['model']
if winner_name=='RandomForest': final_model=search.best_estimator_
else:
    final_model=Pipeline([('prep',preprocess),('model',models[winner_name])])
    final_model.fit(X_train,y_train)
print('Selected by CV:',winner_name)
metrics=score_classifier(winner_name,final_model,X_test,y_test)
display(pd.DataFrame([metrics]).round(3))
print(classification_report(y_test,final_model.predict(X_test),zero_division=0))
ConfusionMatrixDisplay.from_estimator(final_model,X_test,y_test);plt.show()
PrecisionRecallDisplay.from_estimator(final_model,X_test,y_test);plt.show()
RocCurveDisplay.from_estimator(final_model,X_test,y_test);plt.show()


## 11. Explainability: permutation importance
Permutation importance measures the decrease in held-out PR-AUC when one feature is shuffled. It reflects predictive reliance, not causality; correlated features can share importance.


In [ ]:
imp=permutation_importance(final_model,X_test,y_test,n_repeats=3,random_state=SEED,scoring='average_precision',n_jobs=1)
importance=pd.Series(imp.importances_mean,index=X_test.columns).sort_values(ascending=False)
display(importance.to_frame('mean_PR_AUC_drop'))
importance.plot.barh(title='Permutation importance');plt.tight_layout();plt.show()


## 12. Save and reload a complete prediction pipeline
Persist preprocessing + model together. Only load model artifacts you trust; joblib/pickle can execute code during loading.


In [ ]:
(ROOT/'models').mkdir(exist_ok=True)
path=ROOT/'models'/'manufacturing_pipeline.joblib'
dump(final_model,path)
reloaded=load(path)
example=X_test.iloc[[0]]
print('Example prediction:',reloaded.predict(example)[0], 'Probability:',round(reloaded.predict_proba(example)[0,1],4))
(ROOT/'reports').mkdir(exist_ok=True)
cv_results.to_csv(ROOT/'reports'/'manufacturing_cv_comparison.csv',index=False)
print('Saved:',path)


## 13. Limitations and production roadmap
Synthetic data is for learning only. For real projects: obtain consent and governance approval; perform leakage checks; use time/group splits for repeated patients, machines or chargers; evaluate subgroup performance and calibration; measure drift; use CI/CD and secure model registry; monitor operational false alarms and missed events. Healthcare models require independent clinical validation and regulatory review.

**Optional extensions:** ANOVA/Kruskal-Wallis for multi-group comparisons; Fisher exact for small contingency tables; bootstrapped uncertainty; SMOTE within training folds only; XGBoost/LightGBM if separately installed; survival models for time-to-failure; SHAP; cost-sensitive threshold optimization.
